# 02 — Preprocessing

Cleans, resamples, and chronologically partitions each raw water-level series.

**Inputs:** `data/raw/*.parquet`
**Outputs:** `data/processed/{station_id}_train.parquet`, `{station_id}_test.parquet`, and `{station_id}_preprocess_metadata.json`

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
## Setup

from pathlib import Path

from tqdm.auto import tqdm

from src.config import (
    MAX_INTERPOLATION_GAP_HOURS,
    STATION_IDS,
    TEST_FRACTION,
    WEATHER_VARIABLES,
)
from src.fetch_data import summarize_failures
from src.preprocess import (
    preprocess_station,
    split_train_test,
    write_preprocess_artifacts,
)

RAW_DIR = Path("data/raw")
PROCESSED_DIR = Path("data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
failures = {}

## Per-station hourly water level + weather

Each station's water-level history is reindexed to a strict hourly UTC grid;
gaps of at most `MAX_INTERPOLATION_GAP_HOURS` are linearly interpolated and
flagged via `imputed`, longer gaps stay `NaN`. GeoSphere INCA weather is left-joined
onto that grid, since the water-level timeline is the forecasting target. The first
`floor((1 - TEST_FRACTION) * N)` rows are written as training data and the
remainder as a physically sealed test artifact.

In [ ]:
latest_station = None

for station_id in tqdm(STATION_IDS, desc="Preprocessing", unit="station"):
    try:
        station = preprocess_station(
            station_id,
            raw_dir=RAW_DIR,
            max_gap_hours=MAX_INTERPOLATION_GAP_HOURS,
            weather_variables=WEATHER_VARIABLES,
        )
        train, test = split_train_test(station, TEST_FRACTION)
        manifest = write_preprocess_artifacts(
            train,
            test,
            station_id=station_id,
            output_dir=PROCESSED_DIR,
            test_fraction=TEST_FRACTION,
        )
        print(
            f"Saved {len(train):,} train and {len(test):,} test rows for {station_id}"
        )
        latest_station = train
    except Exception as error:  # noqa: BLE001 -- aggregate every station failure
        failures[station_id] = error
        print(f"Failed {station_id}: {error}")

if latest_station is not None:
    latest_station.info()
    display(latest_station.head())
else:
    print("No preprocessed DataFrame is available to display.")

In [ ]:
if failures:
    raise RuntimeError(summarize_failures(failures))

print("Preprocessing completed successfully.")